In [43]:
"""
Streaming Language Modeling Data Pipeline with Hugging Face Datasets
--------------------------------------------------------------------
Goal:
    Demonstrate how to build a *true streaming* LM pipeline that:
    - Processes data without loading the entire dataset into RAM.
    - Tokenizes on the fly.
    - Concatenates text and chunks into fixed-length blocks for LM training.
    - Produces batches ready for training in PyTorch.

Key Teaching Points:
    1. Streaming allows us to work with web-scale corpora.
    2. We still can do grouping/chunking in a rolling fashion.
    3. This approach mimics real-world pipelines for large-scale LM training.
"""
print(__doc__)


Streaming Language Modeling Data Pipeline with Hugging Face Datasets
--------------------------------------------------------------------
Goal:
    Demonstrate how to build a *true streaming* LM pipeline that:
    - Processes data without loading the entire dataset into RAM.
    - Tokenizes on the fly.
    - Concatenates text and chunks into fixed-length blocks for LM training.
    - Produces batches ready for training in PyTorch.

Key Teaching Points:
    1. Streaming allows us to work with web-scale corpora.
    2. We still can do grouping/chunking in a rolling fashion.
    3. This approach mimics real-world pipelines for large-scale LM training.



In [44]:
# !pip install transformers, AutoTokenizer, torch

In [45]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader, get_worker_info
import torch

In [46]:
# ============================================================
# 1. Load the dataset in STREAMING mode
# ============================================================
# Streaming mode returns an IterableDataset — you can iterate over it
# without having all the data in memory at once.
stream_dataset = load_dataset(
    "wikitext", 
    "wikitext-2-raw-v1", 
    split="train", 
    streaming=True
)

In [47]:
# ============================================================
# 2. Initialize the tokenizer
# ============================================================
# For GPT-2, there is no pad token by default, so we set pad_token = eos_token.
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [48]:
# ============================================================
# 3. Tokenization step
# ============================================================
# We do NOT pad/truncate here — we want raw token sequences.
# This keeps flexibility to later concatenate across documents.
def tokenize_function(examples):
    return tokenizer(examples["text"])

# Map tokenization lazily over the streaming dataset
tokenized_stream = stream_dataset.map(tokenize_function, batched=True)

In [49]:
# ============================================================
# 4. Rolling buffer for grouping into fixed-length blocks
# ============================================================
# Because streaming datasets are iterators, we can't look ahead arbitrarily.
# We'll keep a buffer that stores leftover tokens from the previous batch,
# so we can concatenate and chunk consistently.
block_size = 128

def group_texts_streaming(dataset_iter, block_size):
    buffer = []
    for example in dataset_iter:
        buffer.extend(example["input_ids"])
        while len(buffer) >= block_size:
            chunk = buffer[:block_size]
            buffer = buffer[block_size:]
            yield {
                "input_ids": chunk,
                "attention_mask": [1] * block_size
            }


In [ ]:
# ============================================================
# 5. Wrap generator in an IterableDataset
# ============================================================

def make_stream():
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train", streaming=True)
    return ds

class StreamingLMIterableDataset(IterableDataset):
    def __init__(self, stream_factory, block_size=1024, drop_remainder=True):
        super().__init__()
        self.stream_factory = stream_factory
        self.block_size = block_size
        self.drop_remainder = drop_remainder

    def _pack_blocks(self, ids_iter):
        buf = []
        for ids in ids_iter:
            buf.extend(ids)
            while len(buf) >= self.block_size:
                chunk = buf[:self.block_size]
                buf = buf[self.block_size:]
                yield torch.tensor(chunk, dtype=torch.long)
        if not self.drop_remainder and buf:
            pad = [tokenizer.pad_token_id] * (self.block_size - len(buf))
            yield torch.tensor(buf + pad, dtype=torch.long)

    def __iter__(self):
        # Create the stream INSIDE the worker process
        ds = self.stream_factory()
        wi = get_worker_info()
        if wi is not None and wi.num_workers > 1:
            # Proper sharding so each worker sees a distinct slice
            ds = ds.shard(num_shards=wi.num_workers, index=wi.id)

        # Tokenize ON-THE-FLY in the worker (avoid passing tokenized objects across processes)
        tokenized = ds.map(tokenize_function, batched=True)
        ids_iter = (ex["input_ids"] for ex in tokenized)
        return self._pack_blocks(ids_iter)

grouped_iterable_dataset = StreamingLMIterableDataset(make_stream, block_size=1024)

# grouped_iterable_dataset = StreamingLMIterableDataset(tokenized_stream, block_size)

In [51]:
# ============================================================
# 6. Collate function for batches
# ============================================================
# def collate_fn(batch):
#     input_ids = torch.tensor([ex["input_ids"] for ex in batch], dtype=torch.long)
#     attention_mask = torch.tensor([ex["attention_mask"] for ex in batch], dtype=torch.long)
#     return {
#         "input_ids": input_ids,
#         "attention_mask": attention_mask,
#         "labels": input_ids.clone()
#     }
def collate_fn(batch):
    input_ids = torch.stack(batch)                   # [B, T]
    attention_mask = torch.ones_like(input_ids)     # packed → all 1s
    return {
        "input_ids": input_ids,
        "labels": input_ids.clone(),
        "attention_mask": attention_mask
    }

In [52]:
# ============================================================
# 7. DataLoader for streaming data
# ============================================================
train_loader = DataLoader(
    grouped_iterable_dataset,
    batch_size=8,
    collate_fn=collate_fn,
    num_workers=0,              # try 1 or 2 on Windows
    pin_memory=True,
    persistent_workers=False,   # safer in notebooks on Windows
)

for i, b in enumerate(train_loader, 1):
    print("Batch", i, "shape:", b["input_ids"].shape)
    if i == 2: break
# train_loader = DataLoader(grouped_iterable_dataset, batch_size=8, collate_fn=collate_fn, num_workers=2)

Batch 1 shape: torch.Size([8, 1024])
Batch 2 shape: torch.Size([8, 1024])


In [53]:
# ============================================================
# 8. Iterate over a few batches
# ============================================================
print("Sample streaming batches:")
for i, batch in enumerate(train_loader):
    print(f"Batch {i} -> input_ids shape: {batch['input_ids'].shape}")
    if i == 2:
        break

Sample streaming batches:
Batch 0 -> input_ids shape: torch.Size([8, 1024])
Batch 1 -> input_ids shape: torch.Size([8, 1024])
Batch 2 -> input_ids shape: torch.Size([8, 1024])


In [54]:
import time

def measure_throughput(loader, steps=200):
    t0, toks = time.time(), 0
    for i, b in enumerate(loader, 1):
        toks += b["input_ids"].numel()
        if i >= steps: break
    dt = time.time() - t0
    print(f"Steps: {i}, Tokens: {toks:,}, Time: {dt:.2f}s, ~{toks/max(dt,1e-6):,.0f} tokens/sec")

measure_throughput(train_loader, steps=200)


Steps: 200, Tokens: 1,638,400, Time: 17.88s, ~91,634 tokens/sec
